In [1]:
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

DB = '../data/humans_clean.sqlite3'

conn = sqlite3.connect(DB)
conn.execute('PRAGMA cache_size=-1000000')

# Individuals with polity match per century
cur = conn.execute("""
    SELECT (impact_date / 100) * 100 AS century, COUNT(DISTINCT wikidata_id) as cnt
    FROM individuals_cliopatria
    WHERE impact_date BETWEEN -3500 AND 2050
    GROUP BY century ORDER BY century
""")
polity_by_century = {r[0]: r[1] for r in cur.fetchall()}

centuries = sorted(c for c in polity_by_century if -3500 <= c <= 2000 and polity_by_century[c] >= 5)
polity_counts = [polity_by_century[c] for c in centuries]

conn.close()

print(f"{len(centuries)} centuries, {sum(polity_counts):,} individuals assigned to a polity")

50 centuries, 4,778,316 individuals assigned to a polity


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.bar(centuries, polity_counts, width=80, color='#4a4a4a', edgecolor='white',
       linewidth=0.5, alpha=0.85)

ax.set_yscale('log')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_xlabel('Century')
ax.set_ylabel('Number of individuals (log)')
ax.set_xlim(-3600, 2100)

xticks = list(range(-3500, 2500, 500))
ax.set_xticks(xticks)
ax.set_xticklabels([str(x) for x in xticks], fontsize=8)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='both', which='both', length=3)
ax.grid(axis='y', alpha=0.15, linestyle='--')
ax.axvline(x=0, color='#bbbbbb', linestyle=':', linewidth=0.7)

plt.tight_layout()
plt.show()